<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Web scraping con BeautifulSoup**

## *Extraer información estructurada desde HTML*

En este laboratorio aprenderemos a extraer información de páginas web cuando no disponemos de una API adecuada.

La secuencia será:

> **HTML → inspección → selectores → extracción → limpieza → validación → paginación → persistencia**

Trabajaremos con:

- HTML;
- DOM;
- etiquetas;
- atributos;
- CSS selectors;
- enlaces;
- tablas;
- textos;
- BeautifulSoup;
- `requests`;
- errores por cambios de estructura;
- paginación conceptual;
- comparación entre API y scraping.

## Objetivos

Al finalizar deberías poder:

1. Explicar qué es scraping.
2. Entender HTML básico.
3. Identificar etiquetas y atributos.
4. Crear un objeto `BeautifulSoup`.
5. Utilizar `find()` y `find_all()`.
6. Utilizar `select()` y `select_one()`.
7. Extraer texto y atributos.
8. Extraer enlaces.
9. Extraer tablas.
10. Convertir resultados en DataFrame.
11. Detectar cambios de estructura.
12. Limpiar texto.
13. Validar resultados.
14. Diseñar paginación.
15. Comparar API y scraping.
16. Construir un pipeline scraping → DataFrame → archivo.

## 1. Preparación

In [ ]:
from pathlib import Path
import time
import re

import pandas as pd
import matplotlib.pyplot as plt

try:
    import requests
    REQUESTS_DISPONIBLE = True
except ImportError:
    REQUESTS_DISPONIBLE = False

try:
    from bs4 import BeautifulSoup
    BS4_DISPONIBLE = True
except ImportError:
    BS4_DISPONIBLE = False

print(
    "requests:",
    REQUESTS_DISPONIBLE
)

print(
    "BeautifulSoup:",
    BS4_DISPONIBLE
)

# 2. ¿Qué es HTML?

HTML representa la estructura de una página web.

Ejemplo:

```html
<article class="producto">
    <h2>Teclado mecánico</h2>
    <span class="precio">89.90 €</span>
    <a href="/producto/1">Ver producto</a>
</article>
```

Nuestro objetivo es localizar elementos y extraer sus datos.

# 3. Crear HTML local de demostración

In [ ]:
html_demo = '''
<!DOCTYPE html>
<html>
<head>
    <title>Tienda Demo</title>
</head>
<body>

<h1>Catálogo</h1>

<div id="catalogo">

    <article class="producto" data-id="P001">
        <h2 class="nombre">Teclado mecánico</h2>
        <span class="precio">89.90 €</span>
        <span class="stock">12</span>
        <a class="detalle" href="/producto/P001">Ver producto</a>
    </article>

    <article class="producto" data-id="P002">
        <h2 class="nombre">Ratón inalámbrico</h2>
        <span class="precio">39.50 €</span>
        <span class="stock">0</span>
        <a class="detalle" href="/producto/P002">Ver producto</a>
    </article>

    <article class="producto destacado" data-id="P003">
        <h2 class="nombre">Monitor 27 pulgadas</h2>
        <span class="precio">249.00 €</span>
        <span class="stock">5</span>
        <a class="detalle" href="/producto/P003">Ver producto</a>
    </article>

</div>

</body>
</html>
'''

print(
    html_demo[:1200]
)

## 3.1 Parsear HTML

In [ ]:
if BS4_DISPONIBLE:

    soup = BeautifulSoup(
        html_demo,
        "html.parser"
    )

    print(
        soup.title.text
    )

### Interpretación

BeautifulSoup convierte el HTML en una estructura navegable.

# 4. `find()`

In [ ]:
if BS4_DISPONIBLE:

    titulo = soup.find(
        "h1"
    )

    print(
        titulo.text
    )

# 5. `find_all()`

In [ ]:
if BS4_DISPONIBLE:

    productos = soup.find_all(
        "article",
        class_="producto"
    )

    print(
        "Productos:",
        len(
            productos
        )
    )

# 6. Extraer texto

In [ ]:
if BS4_DISPONIBLE:

    primer_producto = productos[0]

    nombre = (
        primer_producto
        .find(
            "h2"
        )
        .get_text(
            strip=True
        )
    )

    precio = (
        primer_producto
        .find(
            "span",
            class_="precio"
        )
        .get_text(
            strip=True
        )
    )

    print(
        nombre,
        precio
    )

# 7. Extraer atributos

In [ ]:
if BS4_DISPONIBLE:

    enlace = primer_producto.find(
        "a",
        class_="detalle"
    )

    print(
        "href:",
        enlace[
            "href"
        ]
    )

    print(
        "data-id:",
        primer_producto.get(
            "data-id"
        )
    )

# 8. CSS selectors

In [ ]:
if BS4_DISPONIBLE:

    nombres = soup.select(
        ".producto .nombre"
    )

    [
        elemento.get_text(
            strip=True
        )
        for elemento
        in nombres
    ]

# 8.1 `select_one()`

In [ ]:
if BS4_DISPONIBLE:

    destacado = soup.select_one(
        ".producto.destacado"
    )

    destacado.get_text(
        " ",
        strip=True
    )

### `find` vs CSS selectors

Ambos enfoques son útiles.

CSS selectors suelen resultar muy expresivos cuando la estructura depende de clases, IDs y jerarquías.

# 9. Extraer todos los productos

In [ ]:
if BS4_DISPONIBLE:

    registros = []

    for producto in soup.select(
        "article.producto"
    ):

        registros.append({
            "id":
                producto.get(
                    "data-id"
                ),

            "nombre":
                producto.select_one(
                    ".nombre"
                ).get_text(
                    strip=True
                ),

            "precio_texto":
                producto.select_one(
                    ".precio"
                ).get_text(
                    strip=True
                ),

            "stock":
                producto.select_one(
                    ".stock"
                ).get_text(
                    strip=True
                ),

            "url":
                producto.select_one(
                    "a.detalle"
                ).get(
                    "href"
                )
        })

    registros

# 9.1 Convertir a DataFrame

In [ ]:
if BS4_DISPONIBLE:

    df_productos = pd.DataFrame(
        registros
    )

    df_productos

# 10. Limpiar precio

In [ ]:
def limpiar_precio(
    texto
):
    texto = str(
        texto
    )

    texto = (
        texto
        .replace(
            "€",
            ""
        )
        .replace(
            ",",
            "."
        )
        .strip()
    )

    return float(
        texto
    )


if BS4_DISPONIBLE:

    df_productos[
        "precio"
    ] = df_productos[
        "precio_texto"
    ].apply(
        limpiar_precio
    )

    df_productos[
        "stock"
    ] = df_productos[
        "stock"
    ].astype(
        int
    )

    df_productos

## 10.1 Visualizar precios

In [ ]:
if BS4_DISPONIBLE:

    plt.figure(
        figsize=(8, 5)
    )

    plt.bar(
        df_productos[
            "nombre"
        ],
        df_productos[
            "precio"
        ]
    )

    plt.ylabel(
        "Precio"
    )

    plt.title(
        "Precio de productos extraídos"
    )

    plt.xticks(
        rotation=20
    )

    plt.tight_layout()
    plt.show()

# 11. Detectar agotados

In [ ]:
if BS4_DISPONIBLE:

    df_productos[
        "agotado"
    ] = (
        df_productos[
            "stock"
        ] == 0
    )

    df_productos

# 12. Provocar cambio de estructura

In [ ]:
html_cambiado = html_demo.replace(
    'class="precio"',
    'class="importe"'
)

if BS4_DISPONIBLE:

    soup_cambiado = BeautifulSoup(
        html_cambiado,
        "html.parser"
    )

    producto = (
        soup_cambiado
        .select_one(
            ".producto"
        )
    )

    precio = (
        producto
        .select_one(
            ".precio"
        )
    )

    print(
        "Selector .precio:",
        precio
    )

### Problema

El scraping depende de la estructura HTML.

Si una clase cambia, el scraper puede dejar de encontrar datos.

# 13. Detectar estructura inesperada

In [ ]:
def texto_obligatorio(
    elemento,
    selector,
    campo
):
    encontrado = (
        elemento
        .select_one(
            selector
        )
    )

    if encontrado is None:
        raise ValueError(
            f"No se encontró {campo} con selector {selector}"
        )

    return encontrado.get_text(
        strip=True
    )


if BS4_DISPONIBLE:

    try:
        texto_obligatorio(
            producto,
            ".precio",
            "precio"
        )

    except ValueError as error:
        print(
            type(error).__name__,
            "->",
            error
        )

### Interpretación

Es mejor fallar con un mensaje claro que devolver silenciosamente datos incompletos.

# 14. Scraper robusto por elemento

In [ ]:
def extraer_producto(
    producto
):
    try:
        return {
            "id":
                producto.get(
                    "data-id"
                ),

            "nombre":
                texto_obligatorio(
                    producto,
                    ".nombre",
                    "nombre"
                ),

            "precio":
                limpiar_precio(
                    texto_obligatorio(
                        producto,
                        ".precio",
                        "precio"
                    )
                ),

            "stock":
                int(
                    texto_obligatorio(
                        producto,
                        ".stock",
                        "stock"
                    )
                ),

            "url":
                producto.select_one(
                    "a.detalle"
                ).get(
                    "href"
                ),

            "estado":
                "ok",

            "error":
                None
        }

    except Exception as error:

        return {
            "id":
                producto.get(
                    "data-id"
                ),

            "nombre":
                None,

            "precio":
                None,

            "stock":
                None,

            "url":
                None,

            "estado":
                "error",

            "error":
                str(
                    error
                )
        }

# 14.1 Ejecutar sobre HTML correcto

In [ ]:
if BS4_DISPONIBLE:

    resultados = [
        extraer_producto(
            producto
        )
        for producto
        in soup.select(
            ".producto"
        )
    ]

    pd.DataFrame(
        resultados
    )

# 14.2 Ejecutar sobre HTML cambiado

In [ ]:
if BS4_DISPONIBLE:

    resultados_cambiados = [
        extraer_producto(
            producto
        )
        for producto
        in soup_cambiado.select(
            ".producto"
        )
    ]

    pd.DataFrame(
        resultados_cambiados
    )

### Interpretación

El scraper continúa procesando y deja trazabilidad del fallo de cada elemento.

# 15. Extraer enlaces

In [ ]:
if BS4_DISPONIBLE:

    enlaces = []

    for enlace in soup.select(
        "a[href]"
    ):

        enlaces.append({
            "texto":
                enlace.get_text(
                    strip=True
                ),

            "href":
                enlace.get(
                    "href"
                )
        })

    pd.DataFrame(
        enlaces
    )

# 16. Extraer tabla HTML

In [ ]:
html_tabla = '''
<table id="ventas">
    <thead>
        <tr>
            <th>Region</th>
            <th>Ventas</th>
            <th>Operaciones</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>Norte</td>
            <td>12000</td>
            <td>50</td>
        </tr>
        <tr>
            <td>Sur</td>
            <td>9500</td>
            <td>42</td>
        </tr>
        <tr>
            <td>Este</td>
            <td>14300</td>
            <td>60</td>
        </tr>
    </tbody>
</table>
'''

if BS4_DISPONIBLE:

    soup_tabla = BeautifulSoup(
        html_tabla,
        "html.parser"
    )

    tabla = soup_tabla.select_one(
        "#ventas"
    )

    cabeceras = [
        th.get_text(
            strip=True
        )
        for th
        in tabla.select(
            "thead th"
        )
    ]

    filas = []

    for tr in tabla.select(
        "tbody tr"
    ):

        filas.append([
            td.get_text(
                strip=True
            )
            for td
            in tr.select(
                "td"
            )
        ])

    df_tabla = pd.DataFrame(
        filas,
        columns=cabeceras
    )

    df_tabla

# 16.1 Convertir tipos

In [ ]:
if BS4_DISPONIBLE:

    df_tabla[
        "Ventas"
    ] = df_tabla[
        "Ventas"
    ].astype(
        float
    )

    df_tabla[
        "Operaciones"
    ] = df_tabla[
        "Operaciones"
    ].astype(
        int
    )

    df_tabla.dtypes

# 17. `pandas.read_html` como alternativa

In [ ]:
try:
    tablas_pd = pd.read_html(
        html_tabla
    )

    print(
        "Tablas detectadas:",
        len(
            tablas_pd
        )
    )

    display(
        tablas_pd[
            0
        ]
    )

except Exception as error:
    print(
        type(error).__name__,
        error
    )

### BeautifulSoup vs `read_html`

- BeautifulSoup: control fino.
- `read_html`: muy cómodo para tablas bien formadas.

# 18. Scraping desde una URL

Cuando existe conexión, el flujo es:

```python
response = requests.get(url, timeout=10)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
```

No necesitamos depender de Internet para aprender selectores, por eso gran parte del laboratorio utiliza HTML local reproducible.

# 18.1 Función de descarga segura

In [ ]:
def descargar_html(
    url,
    timeout=10
):
    if not REQUESTS_DISPONIBLE:
        return None

    try:
        respuesta = requests.get(
            url,
            timeout=timeout,
            headers={
                "User-Agent":
                    "CursoDataGathery/1.0"
            }
        )

        respuesta.raise_for_status()

        return respuesta.text

    except requests.RequestException as error:

        print(
            type(error).__name__,
            "->",
            error
        )

        return None

# 19. Respeto básico antes de scrapear

Antes de automatizar la extracción de un sitio real debemos revisar:

- si existe una API oficial;
- términos de uso;
- `robots.txt`;
- frecuencia razonable de peticiones;
- carga sobre el servidor;
- autenticación;
- tratamiento de datos personales.

El scraping no debe diseñarse como una avalancha de peticiones.

# 20. Introducir espera entre peticiones

In [ ]:
def espera_respetuosa(
    segundos=1
):
    print(
        f"Esperando {segundos} segundos..."
    )

    time.sleep(
        segundos
    )


# Demo corta
espera_respetuosa(
    0.2
)

# 21. Paginación conceptual

In [ ]:
paginas_html = {
    1: '''
        <div class="item">A</div>
        <div class="item">B</div>
    ''',

    2: '''
        <div class="item">C</div>
        <div class="item">D</div>
    ''',

    3: ''
}

paginas_html

# 21.1 Extraer múltiples páginas

In [ ]:
if BS4_DISPONIBLE:

    items = []

    pagina = 1

    while True:

        html = paginas_html.get(
            pagina,
            ""
        )

        if not html.strip():
            break

        soup_pagina = BeautifulSoup(
            html,
            "html.parser"
        )

        encontrados = (
            soup_pagina
            .select(
                ".item"
            )
        )

        if not encontrados:
            break

        items.extend(
            [
                item.get_text(
                    strip=True
                )
                for item
                in encontrados
            ]
        )

        pagina += 1

    items

# 22. Validar número de elementos

In [ ]:
if BS4_DISPONIBLE:

    assert len(
        items
    ) == 4

    print(
        "Paginación validada."
    )

# 23. API vs scraping

In [ ]:
comparativa = pd.DataFrame({
    "criterio": [
        "Estructura",
        "Estabilidad",
        "Formato",
        "Autenticación",
        "Cambios visuales",
        "Recomendación"
    ],

    "API": [
        "Contrato explícito",
        "Habitualmente mayor",
        "JSON/XML",
        "Diseñada para ello",
        "No suele afectar",
        "Preferible si existe"
    ],

    "Scraping": [
        "HTML",
        "Puede cambiar",
        "Texto/DOM",
        "Depende del sitio",
        "Puede romper selectores",
        "Usar cuando sea apropiado"
    ]
})

comparativa

### Regla práctica

> **Si existe una API adecuada, normalmente es preferible a scraping.**

# 24. Pipeline scraping → DataFrame → CSV

In [ ]:
if BS4_DISPONIBLE:

    resultados = [
        extraer_producto(
            producto
        )
        for producto
        in soup.select(
            ".producto"
        )
    ]

    df_scraping = pd.DataFrame(
        resultados
    )

    salida = Path(
        "demo_scraping"
    )

    salida.mkdir(
        exist_ok=True
    )

    ruta_csv = (
        salida
        / "productos.csv"
    )

    df_scraping.to_csv(
        ruta_csv,
        index=False
    )

    print(
        ruta_csv.exists()
    )

    df_scraping

# 25. Auditoría básica

In [ ]:
if BS4_DISPONIBLE:

    auditoria = {
        "elementos_detectados":
            len(
                soup.select(
                    ".producto"
                )
            ),

        "correctos":
            int(
                (
                    df_scraping[
                        "estado"
                    ]
                    == "ok"
                ).sum()
            ),

        "errores":
            int(
                (
                    df_scraping[
                        "estado"
                    ]
                    == "error"
                ).sum()
            )
    }

    auditoria

# 26. Visualizar calidad de extracción

In [ ]:
if BS4_DISPONIBLE:

    valores = [
        auditoria[
            "correctos"
        ],
        auditoria[
            "errores"
        ]
    ]

    plt.figure(
        figsize=(7, 5)
    )

    plt.bar(
        [
            "Correctos",
            "Errores"
        ],
        valores
    )

    plt.ylabel(
        "Elementos"
    )

    plt.title(
        "Calidad del scraping"
    )

    plt.tight_layout()
    plt.show()

# 27. Mini-práctica integrada

Construye un scraper sobre HTML local con:

1. al menos 20 elementos;
2. ID;
3. nombre;
4. precio;
5. disponibilidad;
6. enlace;
7. extracción con CSS selectors;
8. limpieza;
9. tipos numéricos;
10. un selector roto deliberadamente;
11. captura de error por elemento;
12. auditoría;
13. DataFrame;
14. gráfico;
15. CSV final;
16. paginación simulada.

# 28. Checklist

- [ ] ¿Existe una API antes de scrapear?
- [ ] ¿He inspeccionado el HTML?
- [ ] ¿Los selectores son suficientemente específicos?
- [ ] ¿Extraigo texto con `strip=True`?
- [ ] ¿Valido elementos obligatorios?
- [ ] ¿Convierto tipos?
- [ ] ¿Detecto cambios de estructura?
- [ ] ¿Un error individual permite continuar?
- [ ] ¿Registro errores?
- [ ] ¿Gestiono paginación?
- [ ] ¿Uso timeout cuando descargo?
- [ ] ¿Respeto términos y frecuencia?
- [ ] ¿Persisto el resultado?
- [ ] ¿Valido el número extraído?

# 29. Ejercicios finales

### Ejercicio 1
Parsea HTML.

### Ejercicio 2
Usa `find()`.

### Ejercicio 3
Usa `find_all()`.

### Ejercicio 4
Usa `select()`.

### Ejercicio 5
Extrae atributos.

### Ejercicio 6
Extrae enlaces.

### Ejercicio 7
Crea DataFrame.

### Ejercicio 8
Limpia precios.

### Ejercicio 9
Convierte tipos.

### Ejercicio 10
Extrae tabla HTML.

### Ejercicio 11
Prueba `read_html`.

### Ejercicio 12
Rompe un selector.

### Ejercicio 13
Detecta el fallo.

### Ejercicio 14
Haz extracción robusta.

### Ejercicio 15
Simula paginación.

### Ejercicio 16
Añade espera.

### Ejercicio 17
Compara API y scraping.

### Ejercicio 18
Reto: construye un pipeline HTML → extracción → validación → CSV.

# 30. Preguntas de reflexión

1. ¿Qué es scraping?
2. ¿Qué es el DOM?
3. ¿Qué diferencia hay entre tag, clase e ID?
4. ¿Qué aporta CSS selector?
5. ¿Qué diferencia hay entre `find` y `select`?
6. ¿Por qué un scraper puede romperse?
7. ¿Cómo detectarías cambios de estructura?
8. ¿Qué aporta `read_html`?
9. ¿Por qué convertir tipos?
10. ¿Por qué validar el número de elementos?
11. ¿Cómo diseñarías paginación?
12. ¿Por qué preferir una API si existe?
13. ¿Qué consideraciones debes revisar antes de scrapear un sitio real?
14. ¿Cómo conectarías scraping con Excel o PDF?

# 31. Cierre

Ahora sabemos obtener datos desde dos fuentes externas:

```text
API
 ↓
JSON

Web
 ↓
HTML
 ↓
BeautifulSoup
```

El siguiente laboratorio integrará ambas vías en un proyecto:

```text
API / Web
    ↓
procesamiento
    ↓
Excel
    ↓
informe automático
    ↓
configuración
    ↓
variables de entorno
    ↓
Git/GitHub
```